# XVA: The Price After the Price · Episode 14: Capstone: XVA of a cross-currency swap

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The market.** All inputs are illustrative, made up for teaching: the AONIA and BBSW curves of Series 1 (`quotes_illustrative.csv`), and CDS spreads, swaption volatilities and XVA parameters (`quotes_xva_illustrative.csv`). Counterparty A, the AA and BB clients and Our bank are fictional.
>
> Educational material only, not financial, accounting or regulatory advice. Capital and margin rules are simplified; consult the current standards.

1. Setup · 2. The running trade · 3. Two cross-currency swaps · 4. Checked against Series 2 · 5. Last episode's question: where the exposure is · 6. CVA and DVA · 7. FVA · 8. Capital and KVA · 9. The bill · 10. Final checkpoint · 11. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_xva_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_xva_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_xva_illustrative.csv').write_text("""kind,name,tenor,value,unit
cds,Counterparty A,1Y,60,bp
cds,Counterparty A,2Y,75,bp
cds,Counterparty A,3Y,90,bp
cds,Counterparty A,5Y,120,bp
cds,Counterparty A,7Y,135,bp
cds,Counterparty A,10Y,150,bp
cds,Our bank,1Y,25,bp
cds,Our bank,2Y,30,bp
cds,Our bank,3Y,35,bp
cds,Our bank,5Y,45,bp
cds,Our bank,7Y,55,bp
cds,Our bank,10Y,65,bp
cds,AA client,1Y,20,bp
cds,AA client,2Y,25,bp
cds,AA client,3Y,30,bp
cds,AA client,5Y,40,bp
cds,AA client,7Y,48,bp
cds,AA client,10Y,55,bp
cds,BB client,1Y,180,bp
cds,BB client,2Y,220,bp
cds,BB client,3Y,260,bp
cds,BB client,5Y,320,bp
cds,BB client,7Y,350,bp
cds,BB client,10Y,380,bp
swaption,AUD,1Yx4Y,90,bp normal vol
swaption,AUD,2Yx3Y,90,bp normal vol
swaption,AUD,3Yx2Y,89,bp normal vol
swaption,AUD,4Yx1Y,88,bp normal vol
param,recovery,,0.40,fraction
param,hw_mean_reversion,,0.03,per year
param,funding_spread,,45,bp
param,cost_of_capital,,0.10,fraction per year
param,capital_ratio,,0.08,fraction of risk-weighted assets
param,risk_weight AA client,,0.20,fraction (Basel CRE20.43: AA corporate)
param,risk_weight Counterparty A,,0.75,fraction (Basel CRE20.43: BBB corporate)
param,risk_weight BB client,,1.00,fraction (Basel CRE20.43: BB corporate)
param,cva_risk_weight AA client,,0.03,fraction (Basel MAR50.16: industrials IG)
param,cva_risk_weight Counterparty A,,0.03,fraction (Basel MAR50.16: industrials IG)
param,cva_risk_weight BB client,,0.07,fraction (Basel MAR50.16: industrials HY)
param,fx_vol,,0.10,AUD/USD lognormal volatility
param,cva_risk_weight Counterparty W,,0.12,fraction (Basel MAR50.16: financials HY)
param,risk_weight Counterparty W,,1.00,fraction (Basel CRE20.43: BB corporate)
cds,Counterparty W,1Y,150,bp
cds,Counterparty W,2Y,170,bp
cds,Counterparty W,3Y,190,bp
cds,Counterparty W,5Y,220,bp
cds,Counterparty W,7Y,235,bp
cds,Counterparty W,10Y,250,bp
""")
print('wrote quotes_xva_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_sofr_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_sofr_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_sofr_illustrative.csv').write_text("""instrument,label,value,unit
ois,1W,3.88,pct
ois,1M,3.87,pct
ois,2M,3.86,pct
ois,3M,3.85,pct
future,SR3 Dec 2026,96.20,price
future,SR3 Mar 2027,96.30,price
future,SR3 Jun 2027,96.39,price
future,SR3 Sep 2027,96.44,price
ois,2Y,3.66,pct
ois,3Y,3.63,pct
ois,5Y,3.63,pct
ois,7Y,3.70,pct
ois,10Y,3.81,pct
ois,15Y,3.94,pct
ois,20Y,3.98,pct
ois,30Y,3.96,pct
""")
print('wrote quotes_sofr_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_fx_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_fx_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_fx_illustrative.csv').write_text("""instrument,label,value,unit
fx,spot,0.71,rate
fx,ON,-0.098,pips
fx,TN,-0.033,pips
fx,1W,-0.23,pips
fx,1M,-1.1,pips
fx,2M,-2.66,pips
fx,3M,-4.29,pips
fx,6M,-10.68,pips
fx,9M,-19.65,pips
fx,1Y,-30.32,pips
xccy,2Y,14.75,bp
xccy,3Y,14.75,bp
xccy,4Y,15.25,bp
xccy,5Y,15.5,bp
xccy,7Y,15.5,bp
xccy,10Y,15.5,bp
""")
print('wrote quotes_fx_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
valuation_date = "2026-09-25"
aud_quotes_file = "quotes_illustrative.csv"
xva_quotes_file = "quotes_xva_illustrative.csv"
sofr_quotes_file = "quotes_sofr_illustrative.csv"
fx_quotes_file = "quotes_fx_illustrative.csv"
n_pairs = 20000   # antithetic pairs: 40,000 simulated paths
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
cal, dc = ql.Australia(), ql.Actual365Fixed()
out = {"meta": {
    "series": "XVA: The Price After the Price",
    "episode": 14,
    "valuation_date": valuation_date,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

AONIA and BBSW curves (Series 1, `episodes/_shared/aud_curves.py`):

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

The SOFR curve and the AUD/USD market of Series 2 (`sofr_curve.py`, `xccy_curve.py`):

In [ ]:
# SOFR curve used from Series 2 Episode 3 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today (the evaluation date) from the setup cell.
#   SOFR OIS: CFTC Swaps Made Available to Trade chart (USD SOFR OIS): spot T+2, annual fixed and floating
#     legs, ACT/360, payment lag 2 days, New York business days; SOFR fixes on the government securities calendar.
#   3M SOFR futures: CME Submission 18-069: SOFR compounded over the IMM-to-IMM Reference Quarter,
#     price = 100 - rate; named by the month the Reference Quarter starts. Convexity set to zero (a simplification).
USD_CAL = ql.UnitedStates(ql.UnitedStates.Settlement)  # stand-in for New York business days
USD_DC = ql.Actual360()
MONTHS = {m: i for i, m in enumerate(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                                      "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}

def sofr_ois_helper(tenor, quote):
    return ql.OISRateHelper(2, ql.Period(tenor), quote, ql.Sofr(), paymentLag=2,
                            paymentFrequency=ql.Annual, paymentCalendar=USD_CAL,
                            convention=ql.ModifiedFollowing, endOfMonth=True)

def build_sofr_curve(quotes):
    """quotes: rows (instrument, label, value); OIS values in %, futures as prices. Returns curve, quotes, helpers."""
    q, helpers = {}, {}
    for r in quotes.itertuples():
        if r.instrument == "future":  # label like "SR3 Dec 2026"
            q[r.label] = ql.SimpleQuote(r.value)
            _, mon, year = r.label.split()
            helpers[r.label] = ql.SofrFutureRateHelper(ql.QuoteHandle(q[r.label]), MONTHS[mon], int(year),
                                                       ql.Quarterly)
        else:
            q[r.label] = ql.SimpleQuote(r.value / 100)
            helpers[r.label] = sofr_ois_helper(r.label, ql.QuoteHandle(q[r.label]))
    curve = ql.PiecewiseLogLinearDiscount(today, list(helpers.values()), USD_DC)
    curve.enableExtrapolation()
    curve.nodes()  # bootstrap now: the helpers only see the curve once it has been built
    return curve, q, helpers

In [ ]:
# AUD discount curve under USD collateral, used from Series 2 Episode 6 on. Copied verbatim into each notebook by
# make_notebook.py, so every notebook runs on its own. Needs: ql, pd, today, cal, dc and the AUD curve code
# (aud_curves.py) and SOFR curve code (sofr_curve.py) from earlier cells.
#   Short end: AUD/USD FX swaps, overnight to one year (forward = spot + points).
#   Long end: BBSW/SOFR mark-to-market cross-currency basis swaps, two to ten years (AFMA term sheet, 2024):
#     spot T+2 on Sydney and New York days, quarterly; receive 3M BBSW + spread on a constant AUD notional,
#     pay SOFR compounded with no spread on a USD notional reset each quarter to the forward FX rate.
#   Collateral: US dollars, discounted at SOFR (AFMA). The curve solved for is the AUD one.
#   Modelling choice: QuantLib's helper pays coupons and notionals on the same dates. A payment lag of zero
#   puts the notional exchanges on the roll dates, as in the term sheet; the coupons then pay two days early,
#   which moves the fair spread by less than a tenth of a basis point here.
JOINT = ql.JointCalendar(ql.Australia(ql.Australia.Settlement), USD_CAL)
FX_TENORS = ["1W", "1M", "2M", "3M", "6M", "9M", "1Y"]

class Market:
    """All curves of the series: AONIA and BBSW (AUD series), SOFR (Episode 3), AUD under USD collateral."""

    def __init__(self, aud_quotes, sofr_quotes, fx_quotes):
        _, handles, self.aud_q, _ = build_curves(aud_quotes)
        self.aonia, self.bbsw3m = handles["AONIA"], handles["BBSW3M"]
        self.sofr_curve, self.sofr_q, _ = build_sofr_curve(sofr_quotes)
        self.sofr = ql.YieldTermStructureHandle(self.sofr_curve)
        self.tom, self.spot = JOINT.advance(today, 1, ql.Days), JOINT.advance(today, 2, ql.Days)
        fx = fx_quotes[fx_quotes.instrument == "fx"].set_index("label").value
        xc = fx_quotes[fx_quotes.instrument == "xccy"].set_index("label").value
        self.S = ql.SimpleQuote(float(fx["spot"]))
        self.points = {t: ql.SimpleQuote(float(fx[t]) / 1e4) for t in ["ON", "TN"] + FX_TENORS}
        self.basis = {t: ql.SimpleQuote(float(v) / 1e4) for t, v in xc.items()}
        self.near_on, self.near_tn = ql.SimpleQuote(0.0), ql.SimpleQuote(0.0)
        self._set_near()
        h = ql.QuoteHandle
        self.helpers = {
            "ON": ql.FxSwapRateHelper(h(self.points["ON"]), h(self.near_on), ql.Period(1, ql.Days), 0, JOINT,
                                      ql.Following, False, False, self.sofr),
            "TN": ql.FxSwapRateHelper(h(self.points["TN"]), h(self.near_tn), ql.Period(1, ql.Days), 1, JOINT,
                                      ql.Following, False, False, self.sofr)}
        for t in FX_TENORS:
            self.helpers[t] = ql.FxSwapRateHelper(h(self.points[t]), h(self.S), ql.Period(t), 2, JOINT,
                                                  ql.ModifiedFollowing, True, False, self.sofr)
        for t, q in self.basis.items():
            self.helpers["basis " + t] = ql.MtMCrossCurrencyBasisSwapRateHelper(
                h(q), ql.Period(t), 2, JOINT, ql.ModifiedFollowing, True,
                bbsw(3, self.bbsw3m), ql.Sofr(self.sofr), self.sofr,
                False,   # isFxBaseCurrencyCollateralCurrency: AUD is the base currency; collateral is USD
                True,    # isBasisOnFxBaseCurrencyLeg: the spread is on the AUD (BBSW) leg
                False,   # isFxBaseCurrencyLegResettable: the USD notional resets, the AUD one is constant
                ql.Quarterly, 0, ql.Quarterly)
        self.curve = ql.PiecewiseLogLinearDiscount(today, list(self.helpers.values()), dc)
        self.curve.enableExtrapolation()
        self.curve.nodes()
        self.aud_usd = ql.YieldTermStructureHandle(self.curve)

    def _set_near(self):  # today's and tomorrow's rates, built back from spot with the ON and TN points
        self.near_tn.setValue(self.S.value() - self.points["TN"].value())
        self.near_on.setValue(self.S.value() - self.points["TN"].value() - self.points["ON"].value())

    def set_spot(self, value):
        self.S.setValue(value)
        self._set_near()

    def fwd_fx(self, d):
        """AUD/USD outright for value date d: spot x P_aud(spot, d) / P_usd(spot, d)."""
        a, u = self.aud_usd, self.sofr
        return self.S.value() * (a.discount(d) / a.discount(self.spot)) / (u.discount(d) / u.discount(self.spot))

def mtm_swap(m, tenor, spread, aud_notional, coupon_lag=0):
    """Value a BBSW/SOFR mark-to-market basis swap to the party receiving the AUD leg (BBSW + spread).

    Dates as the curve's helpers build them: spot + tenor, rolled backwards quarterly, end-of-month rule.
    Each quarter the USD notional is the AUD notional times the forward FX rate to the period start; notionals
    move on roll dates; coupons are paid coupon_lag business days after each period end (term sheet: 2; the
    curve build: 0). Values are in AUD at today's FX rate. Returns the value, the value of one basis point of
    spread, and one row per period.
    """
    sch = list(ql.MakeSchedule(m.spot, m.spot + ql.Period(tenor), ql.Period(ql.Quarterly), calendar=JOINT,
                               convention=ql.ModifiedFollowing, endOfMonth=True, backwards=True))
    pa, pu = m.aud_usd.discount, m.sofr.discount
    fx_today = m.fwd_fx(today)
    index = bbsw(3, m.bbsw3m)
    pv_aud = -aud_notional * pa(sch[0]) + aud_notional * pa(sch[-1])
    pv_usd, bps, rows = 0.0, 0.0, []
    for i, (a, b) in enumerate(zip(sch[:-1], sch[1:])):
        pay = JOINT.advance(b, coupon_lag, ql.Days) if coupon_lag else b
        tau_a, tau_u = dc.yearFraction(a, b), USD_DC.yearFraction(a, b)
        bbsw_rate = (m.bbsw3m.discount(a) / m.bbsw3m.discount(b) - 1) / tau_a
        sofr_rate = (pu(a) / pu(b) - 1) / tau_u
        fx = m.fwd_fx(a)
        usd_n = aud_notional * fx
        aud_cpn = aud_notional * (bbsw_rate + spread) * tau_a
        usd_cpn = usd_n * sofr_rate * tau_u
        pv_aud += aud_cpn * pa(pay)
        bps += aud_notional * tau_a * pa(pay) * 1e-4
        pv_usd += usd_n * (pu(a) - pu(b)) - usd_cpn * pu(pay)   # receive the USD notional, repay it with SOFR
        rows.append({"period": i + 1, "start": a.ISO(), "end": b.ISO(), "pay": pay.ISO(),
                     "bbsw_pct": bbsw_rate * 100, "aud_rate_pct": (bbsw_rate + spread) * 100, "aud_coupon": aud_cpn,
                     "fx": fx, "usd_notional": usd_n, "sofr_pct": sofr_rate * 100, "usd_coupon": usd_cpn})
    value = pv_aud + pv_usd / fx_today
    return {"value": value, "bps": bps, "pv_aud_leg": pv_aud, "pv_usd_leg_aud": pv_usd / fx_today, "rows": rows}

def par_spread(m, tenor, coupon_lag=0):
    """Spread on the AUD leg that makes a new mark-to-market basis swap worth zero."""
    r = mtm_swap(m, tenor, 0.0, 1.0, coupon_lag)
    return -r["value"] / (r["bps"] * 1e4)

The XVA toolkit used throughout the series (`episodes/_shared/xva.py`):

In [ ]:
# XVA toolkit for Series 4 (XVA: The Price After the Price). Copied verbatim into each notebook, so every
# notebook runs on its own. Needs: ql, np, math, today, cal, dc, and the AUD curve code (aud_curves.py).
#
# Model: one-factor Hull-White on the AONIA curve, r(t) = x(t) + phi(t), dx = -a x dt + sigma dW, fitted to
# today's curve. BBSW forwards = AONIA forwards + a deterministic basis taken from today's curves.

LGD_STANDARD = 0.60  # 40% recovery, the standard assumption for senior unsecured CDS


def credit_curve(spreads_bp, recovery, discount):
    """Bootstrap a flat-hazard credit curve from CDS par spreads {tenor: bp}.

    Standard CDS terms: quarterly premiums on CDS2015 dates, ACT/360, accrual paid on default.
    """
    helpers = [ql.SpreadCdsHelper(s / 1e4, ql.Period(t), 0, ql.WeekendsOnly(), ql.Quarterly, ql.Following,
                                  ql.DateGeneration.CDS2015, ql.Actual360(), recovery, discount)
               for t, s in spreads_bp.items()]
    curve = ql.PiecewiseFlatHazardRate(today, helpers, dc)
    curve.enableExtrapolation()
    curve.nodes()  # bootstrap now, so the helpers are linked and can report implied quotes
    return curve, helpers


def survival(curve, times):
    return np.array([curve.survivalProbability(float(t)) for t in times])


class HullWhite1F:
    """Hand formulas for one-factor Hull-White fitted to a discount curve (Brigo-Mercurio G1++ form)."""

    def __init__(self, curve, a, sigma):
        self.curve, self.a, self.sigma = curve, a, sigma

    def P0(self, t):
        return np.vectorize(lambda u: self.curve.discount(float(u)))(t)

    def B(self, t, T):
        return (1 - np.exp(-self.a * (T - t))) / self.a

    def V(self, t, T):
        a, s, tau = self.a, self.sigma, T - t
        return s**2 / a**2 * (tau + 2 / a * np.exp(-a * tau) - np.exp(-2 * a * tau) / (2 * a) - 1.5 / a)

    def bond(self, t, T, x):
        """Zero-coupon bond price P(t, T) given the state x(t)."""
        return (self.P0(T) / self.P0(t) * np.exp(0.5 * (self.V(t, T) - self.V(0, T) + self.V(0, t))
                                                 - self.B(t, T) * x))

    def forward_measure_x(self, t):
        """Mean and standard deviation of x(t) under the t-forward measure (used to price one exposure date)."""
        a, s = self.a, self.sigma
        mean = -(s**2 / a**2 * (1 - math.exp(-a * t)) - s**2 / (2 * a**2) * (1 - math.exp(-2 * a * t)))
        return mean, s * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))

    def simulate(self, grid, n_pairs, seed):
        """Exact joint simulation of x(t) and its integral on the grid (antithetic pairs).

        Returns X (paths x grid) and D, the discount factor exp(-integral of r) from today to each grid time.
        """
        a, s = self.a, self.sigma
        rng = np.random.default_rng(seed)
        z = rng.standard_normal((n_pairs, len(grid) - 1, 2))
        z = np.concatenate([z, -z])
        X = np.zeros((2 * n_pairs, len(grid)))
        I = np.zeros_like(X)
        for k in range(1, len(grid)):
            dt = grid[k] - grid[k - 1]
            e = math.exp(-a * dt)
            bd = (1 - e) / a
            vx = s**2 / (2 * a) * (1 - e * e)
            vi = s**2 / a**2 * (dt - 2 * bd + (1 - e * e) / (2 * a))
            cxi = s**2 / (2 * a**2) * (1 - e)**2
            L = np.linalg.cholesky([[vx, cxi], [cxi, vi]])
            eps = z[:, k - 1] @ L.T
            X[:, k] = X[:, k - 1] * e + eps[:, 0]
            I[:, k] = I[:, k - 1] + X[:, k - 1] * bd + eps[:, 1]
        D = self.P0(grid) * np.exp(-0.5 * self.V(0, grid) - I)
        return X, D


def calibrate_hw(curve_h, a, vol_quotes, index_months):
    """Fix the mean reversion a and calibrate sigma to co-terminal swaptions quoted in normal vol.

    vol_quotes: [(expiry, tenor, normal vol)]. Single-curve helpers (AONIA projects and discounts).
    """
    model = ql.HullWhite(curve_h, a, 0.01)
    fixed_tenor = ql.Period(f"{index_months}M")
    helpers = []
    for expiry, tenor, vol in vol_quotes:
        h = ql.SwaptionHelper(ql.Period(expiry), ql.Period(tenor), ql.QuoteHandle(ql.SimpleQuote(vol)),
                              bbsw(index_months, curve_h), fixed_tenor, dc, dc, curve_h,
                              ql.BlackCalibrationHelper.RelativePriceError, ql.nullDouble(), 1.0, ql.Normal, 0.0)
        h.setPricingEngine(ql.JamshidianSwaptionEngine(model))
        helpers.append(h)
    model.calibrate(helpers, ql.LevenbergMarquardt(), ql.EndCriteria(1000, 100, 1e-10, 1e-10, 1e-10),
                    ql.NoConstraint(), [], [True, False])
    errors_bp = [(h.impliedVolatility(h.modelValue(), 1e-12, 1000, 1e-4, 0.05) - v) * 1e4
                 for h, (_, _, v) in zip(helpers, vol_quotes)]
    return model.params()[1], model, errors_bp


class SwapLegs:
    """A QuantLib vanilla swap reduced to times and amounts the path pricer needs (all times in years from today)."""

    def __init__(self, swap, ois_curve):
        fl = [ql.as_floating_rate_coupon(c) for c in swap.floatingLeg()]
        fx = [ql.as_fixed_rate_coupon(c) for c in swap.fixedLeg()]
        yf = lambda d: dc.yearFraction(today, d)
        self.ts = np.array([yf(c.accrualStartDate()) for c in fl])  # resets (no fixing lag)
        self.te = np.array([yf(c.date()) for c in fl])  # payments
        self.tau = np.array([c.accrualPeriod() for c in fl])
        f_ois = np.array([(ois_curve.discount(c.accrualStartDate()) / ois_curve.discount(c.date()) - 1) / c.accrualPeriod()
                          for c in fl])
        self.basis = np.array([c.rate() for c in fl]) - f_ois  # BBSW minus AONIA forward, held fixed
        self.tp = np.array([yf(c.date()) for c in fx])
        self.ftau = np.array([c.accrualPeriod() for c in fx])
        self.K = fx[0].rate()
        self.N = fl[0].nominal()
        self.sign = 1 if swap.type() == ql.VanillaSwap.Payer else -1  # payer: receive float, pay fixed
        self.end = self.te[-1]

    def value_at(self, hw, t, x, fixings=None):
        """Swap value at time t for states x. Periods that started on or before t use `fixings` (BBSW set on the
        path); without fixings, t must be a reset date or earlier."""
        v = np.zeros_like(np.asarray(x, dtype=float))
        for i in range(len(self.ts)):
            if self.te[i] <= t + 1e-10:
                continue
            p_end = hw.bond(t, self.te[i], x)
            if self.ts[i] < t - 1e-10:
                flt = fixings[:, i] * self.tau[i] * p_end
            else:
                flt = hw.bond(t, self.ts[i], x) - p_end + self.basis[i] * self.tau[i] * p_end
            v = v + flt
        for j in range(len(self.tp)):
            if self.tp[j] > t + 1e-10:
                v = v - self.K * self.ftau[j] * hw.bond(t, self.tp[j], x)
        return self.sign * self.N * v

    def paths(self, hw, grid, X):
        """Swap value on every path at every grid time (each reset must be a grid time)."""
        gi = {round(t, 10): k for k, t in enumerate(grid)}
        fix = np.zeros((X.shape[0], len(self.ts)))
        for i, t in enumerate(self.ts):
            x = X[:, gi[round(t, 10)]]
            fix[:, i] = (1 / hw.bond(t, self.te[i], x) - 1) / self.tau[i] + self.basis[i]
        return np.column_stack([self.value_at(hw, t, X[:, k], fix) for k, t in enumerate(grid)])


def exposure_grid(legs, horizon_years, per_year=12, extra=()):
    """Monthly (by default) exposure dates plus every reset and payment time, so fixings land on the grid."""
    pts = [np.arange(0, horizon_years + 1e-9, 1 / per_year)] + [l.ts for l in legs] + [l.te for l in legs] + [np.asarray(extra)]
    g = np.unique(np.round(np.concatenate(pts), 10))
    return g[g >= 0]


def mean_se(v):
    """Mean and standard error over antithetic paths (pairs are averaged first)."""
    n = len(v) // 2
    pair = 0.5 * (v[:n] + v[n:])
    return float(pair.mean()), float(pair.std(ddof=1) / math.sqrt(n))


def profile(V, D, idx):
    """Exposure profile at grid indices idx: discounted EE and NEE (both positive numbers), undiscounted EE and
    97.5% PFE."""
    pos, neg = np.maximum(V[:, idx], 0), np.maximum(-V[:, idx], 0)
    return {"EE": (D[:, idx] * pos).mean(axis=0), "NEE": (D[:, idx] * neg).mean(axis=0),
            "EE_undisc": pos.mean(axis=0), "PFE": np.quantile(pos, 0.975, axis=0)}


def default_buckets(credit, times):
    """Probability of default in each interval (times[i-1], times[i]], with times[0] = 0."""
    s = survival(credit, times)
    return np.concatenate([[0.0], s[:-1] - s[1:]])


def cva(ee, times, credit, lgd=LGD_STANDARD):
    """Unilateral CVA = LGD x sum of discounted EE x probability of default in the bucket (EE at bucket end)."""
    return lgd * float(np.dot(ee, default_buckets(credit, times)))


def funding_adjustments(ee, nee, times, spread):
    """Simple symmetric FVA: FCA = spread x sum EE dt, FBA = spread x sum NEE dt (discounted exposures)."""
    dt = np.diff(np.concatenate([[0.0], times]))
    return spread * float(np.dot(ee, dt)), spread * float(np.dot(nee, dt))


def sa_ccr_addon(notional, start, end, t, sf=0.005):
    """SA-CCR interest-rate add-on for one unmargined swap seen from time t: SF x notional x supervisory duration."""
    s, e = max(start - t, 0.0), end - t
    if e <= 0:
        return 0.0
    sd = (math.exp(-0.05 * s) - math.exp(-0.05 * e)) / 0.05
    return sf * notional * sd


def schedule_im(notional, remaining):
    """BCBS-IOSCO standardised initial margin for an interest-rate trade: 1% (<2y), 2% (2-5y), 4% (5y+) of notional."""
    if remaining <= 0:
        return 0.0
    return notional * (0.01 if remaining < 2 else 0.02 if remaining <= 5 else 0.04)


# ---- Collateral (Episode 9 on) ----

def lag_index(grid, idx, days):
    """For each grid index in idx, the grid index closest to `days` calendar days earlier (the margin period of risk)."""
    return np.array([int(np.argmin(np.abs(grid - (grid[k] - days / 365)))) for k in idx])


def collateral_held(V_lagged, threshold=0.0):
    """Two-way variation margin with a threshold, as last called: positive = we hold it, negative = we posted it."""
    return np.maximum(V_lagged - threshold, 0) - np.maximum(-V_lagged - threshold, 0)


# ---- Capital (Episode 11 on) ----

def sa_ccr_ead(V, addon, collateral=0.0, alpha=1.4):
    """SA-CCR exposure at default per path: alpha x (RC + multiplier x add-on), RC = max(V - C, 0), with the
    multiplier for negative value or excess collateral, min(1, 0.05 + 0.95 exp((V - C) / (2 x 0.95 x add-on)))."""
    vc = V - collateral
    if addon <= 0:
        return alpha * np.maximum(vc, 0)
    mult = np.minimum(1.0, 0.05 + 0.95 * np.exp(np.minimum(vc, 0) / (2 * 0.95 * addon)))
    return alpha * (np.maximum(vc, 0) + mult * addon)


def ba_cva_capital(ead, maturity, cva_rw, alpha=1.4, ds=0.65):
    """Reduced BA-CVA capital for one counterparty (one netting set): DS x RW / alpha x M x EAD x DF,
    DF = (1 - exp(-0.05 M)) / (0.05 M). With one counterparty, K_reduced equals its stand-alone CVA."""
    if maturity <= 0:
        return 0.0
    df = (1 - math.exp(-0.05 * maturity)) / (0.05 * maturity)
    return ds * cva_rw / alpha * maturity * ead * df


# ---- FX (Episodes 13-14): lognormal AUD/USD on fixed curves ----

def fx_simulate(fwd0, grid, vol, n_pairs, seed):
    """AUD/USD (USD per AUD) on the grid: FX(t) = F(0, t) exp(vol W(t) - vol^2 t / 2), exact, antithetic pairs."""
    rng = np.random.default_rng(seed)
    dW = rng.standard_normal((n_pairs, len(grid) - 1)) * np.sqrt(np.diff(grid))
    dW = np.concatenate([dW, -dW])
    W = np.concatenate([np.zeros((2 * n_pairs, 1)), np.cumsum(dW, axis=1)], axis=1)
    return np.asarray(fwd0) * np.exp(vol * W - 0.5 * vol**2 * grid), W


class FxCashflows:
    """A trade as AUD and USD cash flows seen by us, valued in USD on fixed curves with a simulated FX rate.

    aud: [(time, AUD amount)], usd: [(time, USD amount)], mtm: [(reset time, pay time, AUD notional, factor)]
    for a USD leg whose notional resets to AUD notional x FX(reset): we receive it at the reset and repay it times
    factor (1 + rate x accrual) at the pay time. P_aud and P_usd map a time to today's discount factor.
    """

    def __init__(self, aud, usd, mtm, P_aud, P_usd):
        self.aud, self.usd, self.mtm, self.Pa, self.Pu = aud, usd, mtm, P_aud, P_usd

    def paths(self, grid, FX):
        gi = {round(t, 10): k for k, t in enumerate(grid)}
        V = np.zeros_like(FX)
        for k, t in enumerate(grid):
            a = sum(c * self.Pa(T) for T, c in self.aud if T > t + 1e-10) / self.Pa(t)
            u = sum(c * self.Pu(T) for T, c in self.usd if T > t + 1e-10) / self.Pu(t)
            v = FX[:, k] * a + u
            for ts, tp, n, f in self.mtm:
                if ts <= t + 1e-10 < tp:
                    v = v - n * FX[:, gi[round(ts, 10)]] * f * self.Pu(tp) / self.Pu(t)
            V[:, k] = v
        return V


# ---- Exposure without simulation (Episode 5 on) ----

QUAD_Z = np.linspace(-9, 9, 8001)
QUAD_W = np.exp(-QUAD_Z**2 / 2) / math.sqrt(2 * math.pi) * (QUAD_Z[1] - QUAD_Z[0])


def ee_at_reset(hw, legs_list, t):
    """Discounted EE of a set of swaps at a date where none has a running fixing (e.g. a common reset date):
    P(0,t) x E[max(sum of values, 0)], with x(t) normal under the t-forward measure. Trapezoid rule on a dense
    grid (Gauss-Hermite converges badly on the kink of max)."""
    m, s = hw.forward_measure_x(t)
    x = m + s * QUAD_Z
    v = sum(l.value_at(hw, t, x) for l in legs_list)
    return float(hw.P0(t) * np.dot(QUAD_W, np.maximum(v, 0)))

## 2. The running trade

> **The running trade.** We pay fixed on a 5-year AUD 100 million swap against 6-month BBSW (semi-annual, the AUD convention beyond 3 years) with Counterparty A, a fictional BBB-rated company, with no collateral agreement. The fixed rate is the par rate, so the trade is worth zero today on the risk-free curves.

Credit curves from CDS spreads (Episode 2), a Hull-White model of the AONIA curve calibrated to swaption volatilities (Episode 3), and 40,000 simulated futures for the trade's value (Episode 4).

In [ ]:
curves, handles, q, _ = build_curves(pd.read_csv(aud_quotes_file))
aonia, aonia_h = curves["AONIA"], handles["AONIA"]
xq = pd.read_csv(xva_quotes_file)
PARAM = {r.name: r.value for r in xq[xq.kind == "param"].itertuples()}
CDS = {who: {r.tenor: r.value for r in g.itertuples()} for who, g in xq[xq.kind == "cds"].groupby("name")}
R = PARAM["recovery"]
LGD = 1 - R
credit = {who: credit_curve(spreads, R, aonia_h)[0] for who, spreads in CDS.items()}
VOLS = [(*r.tenor.split("x"), r.value / 1e4) for r in xq[xq.kind == "swaption"].itertuples()]
A_MR = PARAM["hw_mean_reversion"]
SIGMA, hw_ql, calib_err_bp = calibrate_hw(aonia_h, A_MR, VOLS, 6)
hw = HullWhite1F(aonia, A_MR, SIGMA)

NOTIONAL = 100_000_000
par = vanilla_swap("5Y", 0.04, 6, handles["BBSW6M"], aonia_h, NOTIONAL).fairRate()
trade = vanilla_swap("5Y", par, 6, handles["BBSW6M"], aonia_h, NOTIONAL, receive=False)
legs = SwapLegs(trade, aonia)
annuity = abs(trade.fixedLegBPS()) * 1e4 / NOTIONAL   # PV of 1 a year on the fixed schedule, per unit notional
out["trade"] = {"notional": NOTIONAL, "par_pct": par * 100, "years": 5, "npv": trade.NPV(), "annuity": annuity,
                "sigma_bp": SIGMA * 1e4, "mean_reversion": A_MR, "calib_max_err_bp": max(map(abs, calib_err_bp))}
print(f"par {par:.4%}, NPV {trade.NPV():,.2f}, HW sigma {SIGMA * 1e4:.1f}bp (max vol error {max(map(abs, calib_err_bp)):.2f}bp)")

In [ ]:
m = Market(pd.read_csv(aud_quotes_file), pd.read_csv(sofr_quotes_file), pd.read_csv(fx_quotes_file))
print("AUD/USD spot", m.S.value(), "spot date", m.spot)

## 3. Two cross-currency swaps

With Counterparty A, 5 years, quarterly, from the spot date: we **receive** 3-month BBSW plus a spread on AUD 100
million and **pay** SOFR on US dollars, exchanging notionals at the start and the end.

- **Constant notional**: the USD notional is fixed at the start (AUD 100 million × spot).
- **Mark-to-market (MtM)**: the USD notional resets every quarter to AUD 100 million × the exchange rate then; the
  difference is paid at each reset (Series 2, Episode 6). This is the market standard for AUD/USD.

Each at its own par spread. AUD/USD is lognormal (10%, illustrative); the Series 2 curves are held fixed.

In [ ]:
fx_vol = PARAM["fx_vol"]
N = NOTIONAL
spot_today = m.fwd_fx(today)
S_spot = m.S.value()
mtm_par = par_spread(m, "5Y")
base = mtm_swap(m, "5Y", mtm_par, N)
rows = base["rows"]
iso = lambda s: ql.DateParser.parseISO(s)
yf = lambda d: dc.yearFraction(today, d)
t0, tN = yf(iso(rows[0]["start"])), yf(iso(rows[-1]["end"]))
DATES = {}
def dt_of(t):
    return DATES.get(round(t, 10), today + int(round(t * 365)))
for r in rows:
    for key in ("start", "end", "pay"):
        DATES[round(yf(iso(r[key])), 10)] = iso(r[key])
Pa = lambda t: m.aud_usd.discount(dt_of(t))
Pu = lambda t: m.sofr.discount(dt_of(t))

def flows(spread, mtm):
    aud = [(t0, -N)] + [(yf(iso(r["pay"])), N * (r["bbsw_pct"] / 100 + spread) * dc.yearFraction(iso(r["start"]), iso(r["end"]))) for r in rows] + [(tN, N)]
    if mtm:
        resets = [(yf(iso(r["start"])), yf(iso(r["pay"])), N, 1 + r["sofr_pct"] / 100 * USD_DC.yearFraction(iso(r["start"]), iso(r["end"]))) for r in rows]
        return aud, [], resets
    Nu = N * S_spot
    usd = [(t0, Nu)] + [(yf(iso(r["pay"])), -Nu * r["sofr_pct"] / 100 * USD_DC.yearFraction(iso(r["start"]), iso(r["end"]))) for r in rows] + [(tN, -Nu)]
    return aud, usd, []

def pv_usd(spread, mtm):
    aud, usd, _ = flows(spread, mtm)
    return spot_today * sum(c * Pa(t) for t, c in aud) + sum(c * Pu(t) for t, c in usd)

aud_bps = spot_today * sum(N * dc.yearFraction(iso(r["start"]), iso(r["end"])) * Pa(yf(iso(r["pay"]))) for r in rows) * 1e-4
const_par = -pv_usd(0.0, False) / aud_bps * 1e-4
out["trades"] = {"mtm_par_bp": mtm_par * 1e4, "const_par_bp": const_par * 1e4, "spot": S_spot, "usd_notional": N * S_spot,
                 "periods": len(rows), "annuity": aud_bps * 1e4 / N / spot_today}
out["trades"]

## 4. Checked against Series 2

At today's spot, the path pricer must reproduce the Series 2 MtM swap valuation (`mtm_swap`) at any spread, and both
swaps must be worth zero at their par spreads.

In [ ]:
grid = np.unique(np.round(np.concatenate([np.arange(0, tN + 1e-9, 1 / 12)] + [[t for t, _ in flows(0, False)[0]]]
                                         + [[r[0] for r in flows(0, True)[2]], [r[1] for r in flows(0, True)[2]]]), 10))
fwd0 = np.array([m.fwd_fx(dt_of(t)) if t > 0 else spot_today for t in grid])
FX, Wfx = fx_simulate(fwd0, grid, fx_vol, n_pairs, seed)
def trade_obj(spread, mtm):
    a, u, r = flows(spread, mtm)
    return FxCashflows(a, u, r, Pa, Pu)
test = trade_obj(mtm_par + 0.001, True).paths(grid[:1], FX[:2, :1])[0, 0]
ref = mtm_swap(m, "5Y", mtm_par + 0.001, N)["value"] * spot_today
assert abs(test - ref) < 1.0, (test, ref)
V_m = trade_obj(mtm_par, True).paths(grid, FX)
V_c = trade_obj(const_par, False).paths(grid, FX)
assert abs(V_m[0, 0]) < 1.0 and abs(V_c[0, 0]) < 1.0
out["check"] = {"gap_usd": abs(test - ref)}
out["check"]

## 5. Last episode's question: where the exposure is

Rates are fixed in this model, so all exposure comes from the exchange rate. With a constant notional it's the
final exchange of principal; with MtM resets only one quarter's move is ever outstanding. Discounted EE in AUD:

In [ ]:
D_usd = np.array([Pu(t) for t in grid])
MONx = np.array([k for k, t in enumerate(grid) if abs(t * 12 - round(t * 12)) < 1e-9])
T_X = grid[MONx]
def prof(Vp):
    pos = (D_usd[MONx] * np.maximum(Vp[:, MONx], 0)).mean(axis=0) / spot_today
    neg = (D_usd[MONx] * np.maximum(-Vp[:, MONx], 0)).mean(axis=0) / spot_today
    return pos, neg
ee_c, nee_c = prof(V_c)
ee_m, nee_m = prof(V_m)
pick = int(np.argsort(V_c[:, MONx[24]])[int(0.85 * V_c.shape[0])])
two = T_X <= 2.0 + 1e-9
out["path"] = {"t": T_X[two].tolist(), "const_m": (V_c[pick, MONx][two] / spot_today / 1e6).tolist(),
               "mtm_m": (V_m[pick, MONx][two] / spot_today / 1e6).tolist(),
               "fx2": float(FX[pick, MONx[24]]), "const2": float(V_c[pick, MONx[24]] / spot_today)}
out["exposure"] = {"t": T_X.tolist(), "const_m": (ee_c / 1e6).tolist(), "mtm_m": (ee_m / 1e6).tolist(),
                   "const_peak": float(ee_c.max()), "mtm_peak": float(ee_m.max()), "ratio": float(ee_c.max() / ee_m.max())}
{k: v for k, v in out["exposure"].items() if not isinstance(v, list)}

## 6. CVA and DVA

In [ ]:
cp, bk = credit["Counterparty A"], credit["Our bank"]
xva = {}
for name, ee, nee in (("const", ee_c, nee_c), ("mtm", ee_m, nee_m)):
    xva[name] = {"cva": cva(ee, T_X, cp, LGD), "dva": cva(nee, T_X, bk, LGD)}
xva

## 7. FVA

Funding cost of the collateralised hedge at our funding spread (the funding benefit is left out: it overlaps DVA,
Episode 10).

In [ ]:
SF = PARAM["funding_spread"] / 1e4
for name, ee, nee in (("const", ee_c, nee_c), ("mtm", ee_m, nee_m)):
    xva[name]["fca"] = funding_adjustments(ee, nee, T_X, SF)[0]
{k: v["fca"] for k, v in xva.items()}

## 8. Capital and KVA

SA-CCR for an FX derivative: add-on $= 4\% \times$ notional (in AUD) $\times MF$, and $\text{EAD} = 1.4\,(RC + \text{multiplier}
\times \text{add-on})$ with RC each path's positive value. (Interest-rate add-ons for the two floating legs are left out:
small next to the FX add-on.) Default-risk capital at the BBB risk weight, CVA capital by reduced BA-CVA, cost of
capital 10%. SA-CCR doesn't distinguish a resetting notional, so the add-on is the same for both swaps.

In [ ]:
CR, COC = PARAM["capital_ratio"], PARAM["cost_of_capital"]
rw, crw = PARAM["risk_weight Counterparty A"], PARAM["cva_risk_weight Counterparty A"]
dt_x = np.diff(np.concatenate([[0.0], T_X]))
disc = np.array([Pa(t) for t in T_X])
def kva(Vp):
    caps = []
    for j, k in enumerate(MONx):
        t = grid[k]
        if t >= tN:
            caps.append(0.0); continue
        addon = 0.04 * N * math.sqrt(min(max(tN - t, 10 / 250), 1.0))
        ead = float(sa_ccr_ead(Vp[:, k] / spot_today, addon).mean())
        caps.append(CR * rw * ead + ba_cva_capital(ead, tN - t, crw))
    caps = np.array(caps)
    return COC * float(np.dot(caps * disc, dt_x)), caps[0]
for name, Vp in (("const", V_c), ("mtm", V_m)):
    xva[name]["kva"], xva[name]["cap0"] = kva(Vp)
{k: (v["kva"], v["cap0"]) for k, v in xva.items()}

## 9. The bill

Risk-free value zero (each at its par spread); adjustments in AUD and as basis points a year on the AUD notional.

In [ ]:
ann = out["trades"]["annuity"]
bill = {}
for name in ("const", "mtm"):
    x = xva[name]
    total = -x["cva"] + x["dva"] - x["fca"] - x["kva"]
    bill[name] = {"cva": -x["cva"], "dva": x["dva"], "fca": -x["fca"], "kva": -x["kva"], "total": total,
                  "total_abs": abs(total), "bp": abs(total) / (N * ann) * 1e4, "cap0": x["cap0"],
                  "k": {"cva": -x["cva"] / 1e3, "dva": x["dva"] / 1e3, "fca": -x["fca"] / 1e3, "kva": -x["kva"] / 1e3, "total": total / 1e3}}
rows_tbl = []
for lbl, key in (("CVA", "cva"), ("DVA", "dva"), ("FVA (funding cost)", "fca"), ("KVA", "kva"), ("All-in", "total")):
    rows_tbl.append({"line": lbl, "const": bill["const"][key], "mtm": bill["mtm"][key]})
out["bill"] = {**bill, "rows": rows_tbl, "saving": bill["mtm"]["total"] - bill["const"]["total"],
               "cva_ratio": xva["const"]["cva"] / xva["mtm"]["cva"], "kva_ratio": xva["const"]["kva"] / xva["mtm"]["kva"],
               "quote_const_bp": const_par * 1e4 + bill["const"]["bp"], "quote_mtm_bp": mtm_par * 1e4 + bill["mtm"]["bp"]}
pd.DataFrame(rows_tbl)

## 10. Final checkpoint (answers)

1. *A 3-year swap's EE peaks at AUD 2m; A's yearly default probabilities are 1%, 1.5%, 2%; LGD 60%. Rough CVA with
   EPE AUD 1.5m in each year?* $0.6 \times 1.5\text{m} \times (1\% + 1.5\% + 2\%) =$ AUD 40,500.
2. *Why does daily margin not take exposure to zero?* The margin period of risk: the trade keeps moving between the
   last margin met and close-out.
3. *Why shouldn't you count both DVA and FBA?* Both price your own credit on your negative exposure.
4. *Which is bigger for an uncollateralised BBB corporate swap here, CVA or KVA?* KVA.

In [ ]:
out["checkpoint"] = {"q1": 0.6 * 1.5e6 * (0.01 + 0.015 + 0.02)}
out["checkpoint"]

## 11. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")